# Converged baselines, isiZulu capital split, and low-data runs (kin, lug, swa, zul)

Needs the LoRA sweep first: it reads the chosen LoRA setting from `results/sweep/chosen.json` in the repo.
All runs: AfroXLMR-76L, languages kin, lug, swa, zul, seeds 1 to 3, early stopping on the source dev set.

| block | what | output (repo `results/`) | time on L4 (estimate) |
|---|---|---|---|
| **A** | Baselines trained to convergence: full fine-tuning and tuned LoRA, NER and POS (up to 20 epochs, stop after 3 without dev gain) | `ner_conv`, `pos_conv` | 4 to 6 hours |
| **B** | isiZulu capital split (`uKaunda` fed as `u` + `Kaunda`), tuned LoRA, NER | `ner_capsplit` | about 1 hour |
| **C** | Low-data in-language: 100 and 1,000 training sentences (NER), 100 (POS), full and LoRA (up to 40 epochs, stop after 5) | `ner_n100`, `ner_n1000`, `pos_n100` | about 1 hour |
| **D** (off by default) | Morfessor cuts inside words, and the scrambled-cut control (same number of cuts, random positions), tuned LoRA, NER and POS | `<task>_morfessor_inner_conv`, `<task>_morfessor_scrambled` | about 3 hours |

**Why:** A fixes the undertrained baselines (most NER and POS runs peaked in their last epoch, and LoRA was never
tuned); every later comparison uses A. B tests the isiZulu lead (a third of isiZulu entity words start with a
lowercase prefix before the capital; no source language has any). C tests whether the LoRA gap follows training-set
size (LoRA ties on the small SIB-200 and POS sets but loses on the larger NER sets).

**No Google Drive needed.** Each block's results download as a zip as soon as that block (and task) finishes, so a
disconnect only loses the block that was running. To continue after a disconnect, remove finished blocks from `BLOCKS`
in step 2 and Run all again. With `USE_DRIVE = True`, results go to Drive instead and finished runs are skipped.

**Before you start:** Runtime > Change runtime type > **L4 GPU**. Then Runtime > Run all. Block A is long: Colab Pro
or a session you can leave open is best. You can run B and C first by setting `BLOCKS = ["B", "C", "A"]`.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > L4 GPU, then run again."

## 2. Settings and where results go

In [ ]:
BLOCKS = ["A", "B", "C"]   # remove finished blocks after a disconnect; order sets the run order. Add "D" only if agreed
USE_DRIVE = False          # True: write to Google Drive (survives disconnects, finished runs are skipped)
LANGS = "kin lug swa zul"

from pathlib import Path
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = Path("/content/drive/MyDrive/typology-aware-peft/results")
else:
    RESULTS = Path("/content/results")
RESULTS.mkdir(parents=True, exist_ok=True)
print("Results go to", RESULTS)

## 3. Get the code and the sweep's chosen LoRA setting (from GitHub)

In [ ]:
import os, shutil, json
REPO_URL = "https://github.com/WanjaWhoopie/typology-aware-peft.git"
WORK = Path("/content/typology-aware-peft")
if WORK.exists():
    !git -C {WORK} pull -q
else:
    !git clone -q {REPO_URL} {WORK}
os.chdir(WORK)
for name in ["src/token_baseline.py", "src/token_report.py", "src/sib200_report.py",
             "results/capsplit_seg/zul.json", "results/sweep/chosen.json"]:
    assert Path(name).exists(), f"{name} is not on GitHub yet (run the sweep first and push its results)"
if "D" in BLOCKS:
    assert Path("results/morfessor_scrambled_seg/zul.json").exists(), "scrambled tables are not on GitHub yet"
assert "--train_size" in Path("src/token_baseline.py").read_text(), "token_baseline.py on GitHub is too old; push first"
CHOSEN = json.loads(Path("results/sweep/chosen.json").read_text())
LORA = {t: f"--lora_r {CHOSEN[t]['lora_r']} --lora_targets {CHOSEN[t]['lora_targets']}" for t in ["ner", "pos"]}
print("Chosen LoRA settings:", LORA)
!git log --oneline -1

## 4. Install packages

Same pinned versions as the SIB-200 run. Colab now ships an old `torchao` that makes peft crash when it
builds a LoRA model (this stopped every LoRA run on 2 Nov), so it is removed; nothing here uses it.

In [ ]:
!pip install -q transformers==4.57.2 datasets==4.4.1 peft==0.21.2 scikit-learn
!pip uninstall -y -q torchao
import transformers, datasets, peft
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| peft", peft.__version__)

## 5. Smoke test (about 2 minutes)

Tiny runs on the small base model with the new options (early stopping, few-shot subset, capital split).

In [ ]:
!python src/token_baseline.py --task ner --smoke --model Davlan/afro-xlmr-base --patience 1 --train_size 100 --out /content/smoke_a
assert _exit_code == 0, "smoke test failed; see the error above"
!python src/token_baseline.py --task ner --smoke --model Davlan/afro-xlmr-base --segment results/capsplit_seg --seg_mode word --out /content/smoke_b
assert _exit_code == 0, "smoke test failed; see the error above"

## 6. Run the blocks

Each (block, task) downloads a zip when it finishes: `ner_conv_results.zip`, `pos_conv_results.zip`,
`ner_capsplit_results.zip`, `ner_n100_results.zip`, `ner_n1000_results.zip`, `pos_n100_results.zip`.
**Out of memory:** add `--bs 8` to the command in `JOBS` and note it.

In [ ]:
from google.colab import files

def run(task, name, extra, methods="full lora"):
    out = RESULTS / name
    # LoRA settings from the sweep; full fine-tuning ignores them
    !python src/token_baseline.py --task {task} --methods {methods} --langs {LANGS} {LORA[task]} {extra} --out {out}
    !python src/token_report.py --task {task} --dir {out} > /dev/null
    if not USE_DRIVE:
        shutil.make_archive(f"/content/{name}_results", "zip", out)
        files.download(f"/content/{name}_results.zip")

JOBS = {
    "A": [("ner", "ner_conv", "--epochs 20 --patience 3", "full lora"),
          ("pos", "pos_conv", "--epochs 20 --patience 3", "full lora")],
    "B": [("ner", "ner_capsplit", "--epochs 20 --patience 3 --segment results/capsplit_seg --seg_mode word", "lora")],
    "C": [("ner", "ner_n100", "--epochs 40 --patience 5 --train_size 100", "full lora"),
          ("ner", "ner_n1000", "--epochs 40 --patience 5 --train_size 1000", "full lora"),
          ("pos", "pos_n100", "--epochs 40 --patience 5 --train_size 100", "full lora")],
    "D": [(t, f"{t}_{name}", f"--epochs 20 --patience 3 --segment results/{seg} --seg_mode inner", "lora")
          for t in ("ner", "pos") for name, seg in (("morfessor_inner_conv", "morfessor_seg"),
                                                    ("morfessor_scrambled", "morfessor_scrambled_seg"))],
}
for b in BLOCKS:
    for task, name, extra, methods in JOBS[b]:
        print(f"===== block {b}: {name} ({methods}) =====")
        run(task, name, extra, methods)

## 7. Check the runs

In [ ]:
for name in sorted(p.name for p in RESULTS.iterdir() if (p / "runs.jsonl").exists()):
    runs = [json.loads(l) for l in (RESULTS / name / "runs.jsonl").read_text().splitlines() if l.strip()]
    late = sum(r["best_epoch"] == r["epochs"] for r in runs)
    bad = [f"{r['method']} {r['src']} seed {r['seed']}" for r in runs if r["collapsed"]]
    print(f"{name}: {len(runs)} runs; mean best epoch {sum(r['best_epoch'] for r in runs) / len(runs):.1f}; "
          f"still improving at the epoch limit: {late}; collapsed: {bad or 'none'}")

## 8. Done

Leave the zips in Downloads and tell Claude. They unzip into `results/` in the repo.
If a run collapsed, tell Claude before rerunning anything.